# CU / Genotek · Colorectal cancer baseline
Predict case (1) versus control (0) from stool taxon abundances. [Wirbel et al. (2019)](https://doi.org/10.1038/s41591-019-0406-6) motivate checking whether signals transfer between cohorts. Here the primary estimate measures discrimination among new samples from the same five cohorts. It does not establish diagnostic accuracy in a new population.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

SEED = 42
DATA = Path(os.environ.get("CU_DATA_DIR", "data"))
NUM = ["age", "bmi", "library_size"]
CAT = ["gender", "cohort"]
EPS = 1e-6
train = pd.read_csv(DATA / "train.csv", dtype={"sample_id": str})
assert train.sample_id.notna().all() and train.sample_id.is_unique
assert set(train.label) == {0, 1}
train = train.sort_values("sample_id").reset_index(drop=True)
taxa = sorted(k for k in train if "mOTU" in k)
assert taxa and set(train) == set(taxa + NUM + CAT + ["sample_id", "label"])
features = taxa + NUM + CAT
X, y = train[features], train.label
strata = train.cohort + "/" + y.astype(str)
dev, hold = train_test_split(np.arange(len(train)), test_size=0.2,
                             random_state=SEED, stratify=strata)
assert not set(dev).intersection(hold)
split = pd.Series("development", index=train.index)
split.iloc[hold] = "holdout"
display(pd.crosstab([split, train.cohort], train.label, margins=True))

**Exploration.** After sorting by ID for stable assignment, reserve 20% of each cohort/label mix for a seeded holdout (83 samples total), leaving 332 for development. The split table gives the exact class counts. No samples are excluded. Beyond split counts, summaries use development data. Differences in age, sex, BMI, sequencing depth or cohort composition can confound microbial associations.

In [ ]:
d = train.iloc[dev]
print("Development class counts and cohort composition")
display(pd.crosstab(d.cohort, d.label).assign(case_fraction=d.groupby("cohort").label.mean()))
print("Missing metadata by field")
display(d[NUM + CAT].isna().sum().rename("missing"))
print("Median numeric metadata by class and sex counts by class")
display(d.groupby("label")[NUM].median())
display(pd.crosstab(d.gender, d.label))
abundances = d[taxa]
print(f"{len(taxa)} taxa; {(abundances == 0).to_numpy().mean():.2%} zero entries")
display(pd.DataFrame({"detected_taxa": abundances.gt(0).sum(axis=1),
                      "sum_of_abundances": abundances.sum(axis=1)}).describe())

**Selection.** Compare raw fractions, natural log with a fixed pseudocount of 10⁻⁶, row-centered log ratios using the same pseudocount, and detection (>0). Every candidate uses L2 logistic regression. Numeric values are median-imputed and standardized; categorical values are mode-imputed and one-hot encoded. All fitted steps train inside each fold. There is no supervised feature selection. Choose the largest mean five-fold ROC AUC on development data; exact ties favor smaller C, microbial-only features, then the listed transformation order.

In [ ]:
def transform_taxa(values, representation):
    if (values < 0).any():
        raise ValueError("Expected nonnegative taxon abundances")
    if representation == "raw":
        return values
    if representation == "presence":
        return (values > 0).astype(float)
    logs = np.log(values + EPS)
    if representation == "log":
        return logs
    if representation == "clr":
        return logs - logs.mean(axis=1, keepdims=True)
    raise ValueError(representation)


def build_model(representation, metadata, C):
    micro = make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        FunctionTransformer(transform_taxa, kw_args={"representation": representation}),
        StandardScaler())
    blocks = [("microbial", micro, taxa)]
    if metadata:
        blocks.extend([
            ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
            ("categorical", make_pipeline(SimpleImputer(strategy="most_frequent"),
                                          OneHotEncoder(handle_unknown="ignore")), CAT)])
    return Pipeline([("features", ColumnTransformer(blocks)),
                     ("model", LogisticRegression(C=C, penalty="l2", solver="liblinear",
                                                  max_iter=2000, random_state=SEED))])

In [ ]:
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(inner.split(X.iloc[dev], strata.iloc[dev]))
rows = []
with threadpool_limits(limits=1):
    for strength in (0.01, 0.1, 1.0):
        for metadata in (False, True):
            for representation in ("raw", "log", "clr", "presence"):
                model = build_model(representation, metadata, strength)
                aucs = cross_val_score(model, X.iloc[dev], y.iloc[dev], cv=folds,
                                       scoring="roc_auc", n_jobs=1, error_score="raise")
                rows.append(dict(representation=representation, metadata=metadata, C=strength,
                                 cv_mean=aucs.mean(), cv_sd=aucs.std(ddof=1),
                                 **{f"fold_{i}": a for i, a in enumerate(aucs, 1)}))
results = pd.DataFrame(rows)
winner = rows[int(results.cv_mean.argmax())]
configuration = {key: winner[key] for key in ("representation", "metadata", "C")}
print(results.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("Selected:", configuration)
print(f"Selected mean CV ROC AUC: {winner['cv_mean']:.4f}")

**Held-out evaluation.** Score the selected configuration once using continuous case probabilities. Per-cohort AUC uses pairs within each cohort and is reported only when both classes occur. Small cohort counts limit precision. Fold standard deviation measures variation across splits.

In [ ]:
with threadpool_limits(limits=1):
    evaluated = build_model(**configuration).fit(X.iloc[dev], y.iloc[dev])
    prob = evaluated.predict_proba(X.iloc[hold])[:, 1]
held = train.iloc[hold][["cohort", "label"]].assign(probability=prob)
holdout_auc = roc_auc_score(held.label, held.probability)
print(f"Holdout ROC AUC: {holdout_auc:.4f}")
diagnostics = []
for cohort, group in held.groupby("cohort"):
    pos = int(group.label.sum())
    auc = roc_auc_score(group.label, group.probability) if 0 < pos < len(group) else np.nan
    diagnostics.append(dict(cohort=cohort, controls=len(group)-pos, cases=pos, auc=auc))
print(pd.DataFrame(diagnostics).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

**Submission.** Refit the locked configuration on all labeled samples. Apply the fixed 0.5 probability threshold to produce the required binary labels. Preserve the order of the test file.

In [ ]:
with threadpool_limits(limits=1):
    final_model = build_model(**configuration).fit(X, y)
    test = pd.read_csv(DATA / "test.csv", dtype={"sample_id": str})
    assert test.sample_id.is_unique and test.sample_id.notna().all()
    assert set(test) == set(features + ["sample_id"])
    predictions = (final_model.predict_proba(test[features])[:, 1] >= 0.5).astype(int)
submission = test[["sample_id"]].assign(label=predictions)
assert submission.sample_id.equals(test.sample_id) and submission.label.isin([0, 1]).all()

In [ ]:
display({"text/plain": submission.to_string(index=False),
         "text/html": submission.to_html(index=False),
         "text/csv": submission.to_csv(index=False)}, raw=True)